# Model 08: integrated pedigree + genome

This notebook puts genealogy and autosomal founder-DNA transmission in the **same finite population**.

For multiple founders, genealogy is tracked separately for each founder. Genetic segments are tagged collectively as DNA from the founder set.

In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_DIR = None
if IN_COLAB:
    REPO_DIR = Path('/content/Evolution-Creation')
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git','clone','-q','https://github.com/vafaei-ar/Evolution-Creation.git',str(REPO_DIR)],check=True)
    else:
        subprocess.run(['git','-C',str(REPO_DIR),'fetch','-q','origin','main'],check=True)
        subprocess.run(['git','-C',str(REPO_DIR),'checkout','-q','main'],check=True)
        subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[dev]'],check=True)
else:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / 'src' / 'evolution_creation').exists():
            REPO_DIR = candidate
            break

if REPO_DIR is not None:
    src_path = str(REPO_DIR / 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

print('Environment ready:', REPO_DIR if REPO_DIR is not None else 'using installed Python environment')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from evolution_creation.pedigree_genome import (
    simulate_pedigree_genome,
    simulate_pedigree_genome_replicates,
)


## One integrated population

The main curves answer four different questions: descent from at least one founder, descent from every founder, any surviving founder-set autosomal DNA, and at least one segment above the chosen cM threshold.

In [ ]:
population_size=widgets.IntSlider(value=200,min=30,max=500,step=10,description='Population')
founder_count=widgets.IntSlider(value=2,min=1,max=4,step=1,description='Founders')
max_generations=widgets.IntSlider(value=25,min=5,max=50,step=1,description='Generations')
threshold_cm=widgets.FloatSlider(value=6.0,min=0.0,max=20.0,step=0.5,description='Threshold cM')
distinct_parents=widgets.Checkbox(value=True,description='Distinct parents')
seed=widgets.IntText(value=3,description='Seed')
display(population_size,founder_count,max_generations,threshold_cm,distinct_parents,seed)

In [ ]:
def run_one(_=None):
    result=simulate_pedigree_genome(
        population_size=population_size.value,
        max_generations=max_generations.value,
        founder_count=founder_count.value,
        detectable_threshold_cm=threshold_cm.value,
        distinct_parents=distinct_parents.value,
        seed=seed.value,
    )
    g=result.generations
    fig,ax=plt.subplots(figsize=(10,5))
    ax.plot(g,result.any_founder_descendant_fraction,label='descendant of >=1 founder')
    ax.plot(g,result.all_founders_descendant_fraction,label='descendant of all founders')
    ax.plot(g,result.genetic_carrier_fraction,label='any founder-set DNA')
    ax.plot(g,result.detectable_carrier_fraction,label=f'>= {threshold_cm.value:g} cM founder segment')
    ax.set(xlabel='Generation',ylabel='Population fraction',ylim=(0,1.02))
    ax.legend(ncol=2); plt.show()

    fig,ax=plt.subplots(figsize=(10,5))
    ax.stackplot(
        g,
        result.not_all_founders_fraction,
        result.all_founders_no_dna_fraction,
        result.all_founders_subdetectable_fraction,
        result.all_founders_detectable_fraction,
        labels=['not descended from all founders','all founders, no DNA','all founders, sub-threshold DNA','all founders, detectable DNA'],
    )
    ax.set(xlabel='Generation',ylabel='Population fraction',ylim=(0,1))
    ax.legend(loc='upper left',fontsize=9); plt.show()

    fig,ax=plt.subplots(figsize=(10,4))
    ax.semilogy(g,np.maximum(result.mean_founder_dna_fraction,1e-12),label='population mean founder-set DNA fraction')
    ax.axhline(founder_count.value/population_size.value,linestyle='--',label='initial founder fraction F/N')
    ax.set(xlabel='Generation',ylabel='Mean fraction of diploid autosomal genome')
    ax.legend(); plt.show()

    print('At-least-one-founder fixation:',result.any_founder_fixation_generation)
    print('All-founders universal generation:',result.all_founders_universal_generation)
    print('Founder-set genetic extinction:',result.genetic_extinction_generation)
    print('Ghost generation:',result.ghost_generation)

run_button=widgets.Button(description='Run integrated model',button_style='primary')
run_button.on_click(run_one)
display(run_button)
run_one()

## Replicate variability

Finite populations are stochastic. A founder lineage can disappear early, one member of a founder pair can disappear while the other survives, and genetic ancestry can drift. The replicate panel should therefore be read alongside any single trajectory.

In [ ]:
replicates=widgets.IntSlider(value=10,min=2,max=40,step=1,description='Replicates')
display(replicates)

def run_replicates(_=None):
    summary=simulate_pedigree_genome_replicates(
        population_size=population_size.value,
        max_generations=max_generations.value,
        founder_count=founder_count.value,
        detectable_threshold_cm=threshold_cm.value,
        distinct_parents=distinct_parents.value,
        replicates=replicates.value,
        seed=seed.value,
    )
    g=summary.generations
    fig,ax=plt.subplots(figsize=(10,5))
    ax.plot(g,summary.mean_all_founders_descendant_fraction,label='mean: descendants of all founders')
    ax.plot(g,summary.mean_genetic_carrier_fraction,label='mean: any founder DNA')
    ax.plot(g,summary.mean_detectable_carrier_fraction,label='mean: detectable founder DNA')
    ax.set(xlabel='Generation',ylabel='Mean population fraction',ylim=(0,1.02))
    ax.legend(); plt.show()

    fig,ax=plt.subplots(figsize=(8,4))
    x=np.arange(summary.replicates)
    ax.scatter(x,summary.final_genetic_carrier_fraction,label='any founder DNA')
    ax.scatter(x,summary.final_detectable_carrier_fraction,label='detectable founder DNA')
    ax.set(xlabel='Replicate',ylabel='Final population fraction',ylim=(0,1.02))
    ax.legend(); plt.show()

    print(f'All founders universal by final generation: {summary.all_founders_universal_probability:.1%}')
    print(f'Ghost founder-set event by final generation: {summary.ghost_probability_by_final:.1%}')
    print(f'Median universal generation among reached runs: {summary.median_all_founders_universal_generation:.1f}')

replicate_button=widgets.Button(description='Run replicates')
replicate_button.on_click(run_replicates)
display(replicate_button)

## Interpretation

A universal genealogical founder does not imply universal autosomal genetic contribution. Conversely, population-scale founder DNA does not simply follow the single-path $2^{-k}$ curve because the same founder can reach one person through multiple pedigree paths.

The current model is deliberately panmictic and constant-size. Historical claims require adding migration, endogamy, changing population size, and empirically constrained connectivity.